# Titanic Survival Classification

**Project question:** Which passenger characteristics are associated with survival, and how accurately can a simple model classify unseen passengers?

This notebook uses clear, reproducible Python and focuses on interpretation rather than unnecessarily complex modelling. Run the cells from top to bottom.

## 1 Setup

Import the packages, define paths and create the reusable analysis functions.

In [1]:
"""Predict Titanic survival with clear, leakage-safe pipelines."""

from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


ROOT = Path.cwd()
DATA_PATH = ROOT / "titanic.csv"
OUTPUT_DIR = ROOT / "outputs"
RANDOM_STATE = 42


def make_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline(
        [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    )
    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [("numeric", numeric_pipeline, numeric_features),
         ("categorical", categorical_pipeline, categorical_features)]
    )


def main() -> None:
    OUTPUT_DIR.mkdir(exist_ok=True)
    data = pd.read_csv(DATA_PATH)
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

    numeric = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
    categorical = ["Pclass", "Sex", "Embarked", "IsAlone"]
    X = data[numeric + categorical]
    y = data["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )

    models = {
        "Majority baseline": DummyClassifier(strategy="most_frequent"),
        "Logistic regression": LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE),
        "Decision tree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=RANDOM_STATE),
    }
    comparison = []
    fitted_models = {}
    for name, estimator in models.items():
        pipeline = Pipeline(
            [("preprocessing", make_preprocessor(numeric, categorical)), ("model", estimator)]
        )
        cv_auc = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="roc_auc").mean()
        pipeline.fit(X_train, y_train)
        comparison.append({"model": name, "mean_cv_roc_auc": round(cv_auc, 3)})
        fitted_models[name] = pipeline

    comparison_df = pd.DataFrame(comparison).sort_values("mean_cv_roc_auc", ascending=False)
    comparison_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
    best_name = comparison_df.iloc[0]["model"]
    best_model = fitted_models[best_name]
    predictions = best_model.predict(X_test)
    probabilities = best_model.predict_proba(X_test)[:, 1]

    metrics = {
        "rows": int(len(data)),
        "selected_model": best_name,
        "holdout_accuracy": round(accuracy_score(y_test, predictions), 3),
        "holdout_f1": round(f1_score(y_test, predictions), 3),
        "holdout_roc_auc": round(roc_auc_score(y_test, probabilities), 3),
    }
    (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

    sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Titanic survival - {best_name}")
    plt.xlabel("Predicted survival")
    plt.ylabel("Actual survival")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160)
    plt.close()

    survival_by_group = data.groupby(["Sex", "Pclass"], as_index=False)["Survived"].mean()
    survival_by_group.to_csv(OUTPUT_DIR / "survival_by_sex_and_class.csv", index=False)
    print(comparison_df.to_string(index=False))
    print(json.dumps(metrics, indent=2))



## 2 Inspect the data

Check the number of records, the first rows and the most important quality or balance information.

In [2]:
data = pd.read_csv(DATA_PATH)
print(f'Rows: {len(data):,}')
display(data.head())
display(data.isna().sum().sort_values(ascending=False).head())

Rows: 891


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
dtype: int64

## 3 Run the analysis

This runs the full tested workflow and saves reproducible metrics and charts in the `outputs` folder.

In [3]:
main()

              model  mean_cv_roc_auc
Logistic regression            0.856
      Decision tree            0.849
  Majority baseline            0.500
{
  "rows": 891,
  "selected_model": "Logistic regression",
  "holdout_accuracy": 0.816,
  "holdout_f1": 0.74,
  "holdout_roc_auc": 0.85
}


## 4 Review the main results

In [4]:
import json
results = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
display(pd.Series(results, name='value').to_frame())

,value
rows,891
selected_model,Logistic regression
holdout_accuracy,0.816
holdout_f1,0.74
holdout_roc_auc,0.85


## 5 Conclusion

Logistic regression provides the strongest cross-validated ROC-AUC while remaining easy to explain. The pipeline also keeps imputation, scaling and one-hot encoding inside validation to prevent leakage.

The README summarises the result for recruiters, while this notebook shows how it was produced.